 #  Documentación del Dataset: Simulación de Consumo Energético

## 1. Descripción General
Este pipeline de generación de datos crea un conjunto de datos sintético y realista diseñado específicamente para tareas de Análisis Exploratorio de Datos (EDA) y entrenamiento de modelos de Machine Learning (Clasificación Multiclase).

El script simula el comportamiento energético de miles de viviendas, basándose en características estructurales, demográficas, de equipamiento y factores climáticos. Además, incluye un módulo de inyección de imperfecciones controladas para habilitar la práctica de *Data Cleaning* y manejo de valores atípicos (*Outliers*).

## 2. Especificaciones Técnicas
*   **Volumen Base:** 10,000 registros únicos.
*   **Volumen Final:** ~10,100 registros (incluye un 1% de duplicados estructurales inyectados).
*   **Variable Objetivo (*Target*):** `categoria` (Eficiencia Energética: *Efficient, Moderate, Inefficient*).
*   **Reproducibilidad:** Semilla aleatoria (SEED) fijada en `42` para garantizar que los resultados sean consistentes en cada ejecución.

---

## 3. Lógica de Simulación Estadística
Para garantizar el realismo de los datos y evitar distribuciones uniformes planas, se aplicaron las siguientes reglas matemáticas:
*   **Distribuciones Log-normales:** Utilizadas para los `metros_cuadrados` y el `ingreso_mensual`, simulando la realidad económica donde la mayoría se concentra en valores medios-bajos, con una "cola larga" de propiedades/ingresos de gran magnitud.
*   **Distribución de Poisson:** Empleada para variables de conteo discreto (ej. `habitaciones`, `aires_acondicionados`, `cantidad_personas`), asegurando que no existan valores negativos y respetando proporciones lógicas (ej. 1 habitación por cada ~35m²).
*   **Asociación de Variables:** El ecosistema de datos está fuertemente correlacionado. Por ejemplo, una casa antigua tiene mayor probabilidad de tener un mal aislamiento térmico, y los ingresos mensuales condicionan la cantidad de electrodomésticos.

---

## 4. Diccionario de Datos

A continuación, se detallan las variables generadas, agrupadas por su módulo de origen:

###  Características de la Vivienda (`crear_viviendas`)
*   **`id`** (Entero): Identificador único del registro.
*   **`tipo_vivienda`** (Categórica): Clasificación del inmueble ('Casa', 'Departamento', 'Pequeño Comercio').
*   **`metros_cuadrados`** (Flotante): Superficie del inmueble acotada entre 30m² y 450m².
*   **`habitaciones`** (Entero): Cantidad de habitaciones (condicionado por los metros cuadrados).
*   **`baños`** (Entero): Cantidad de baños (condicionado por las habitaciones).
*   **`antiguedad_vivienda`** (Entero): Años de antigüedad del inmueble (Distribución triangular, moda = 15).
*   **`aislamiento`** (Categórica): Calidad del aislamiento térmico ('Poor', 'Average', 'Good', 'Excellent').
*   **`eficiencia_construccion`** (Categórica): Calificación constructiva ('E', 'D', 'C', 'B', 'A').
*   **`paneles_solares`** (Booleano): Indica si la propiedad posee generación solar.

###  Demografía y Ocupación (`crear_habitantes`)
*   **`cantidad_personas`** (Entero): Habitantes de la propiedad (relacionado con el número de habitaciones).
*   **`trabajo_remoto`** (Booleano): Indica si al menos un habitante hace *Home Office*.
*   **`horas_en_casa`** (Flotante): Promedio de horas diarias de ocupación de la vivienda.
*   **`ingreso_mensual`** (Flotante): Ingreso estimado del hogar en USD (correlacionado con el tamaño del inmueble).

###  Equipamiento y Climatización (`crear_equipamiento`)
*   **`aires_acondicionados`** (Entero): Cantidad de equipos de aire acondicionado (0 a 5).
*   **`heladeras`** (Entero): Conteo de heladeras/refrigeradores.
*   **`televisores`** (Entero): Conteo de televisores en la vivienda.
*   **`computadoras`** (Entero): Conteo de computadoras personales o de escritorio.
*   **`lavadoras`** (Entero): Conteo de máquinas lavadoras.
*   **`secadoras`** (Entero): Conteo de máquinas secadoras de ropa.
*   **`cantidad_equipos`** (Entero): Suma total de dispositivos eléctricos, incluyendo pequeños electrodomésticos.
*   **`calefaccion`** (Booleano): Existencia de sistema térmico de invierno.
*   **`tipo_calefaccion`** (Categórica): Tipo de sistema térmico de invierno ('Ninguna', 'Gas', 'Losa Radiante', 'Eléctrica').
*   **`tipo_iluminacion`** (Categórica): Tecnología de iluminación predominante ('LED', 'Mixta', 'Incandescente').
*   **`electrodomesticos_eficientes`** (Flotante): Porcentaje del parque de electrodomésticos considerado de alta eficiencia (%).

###  Consumo Energético (`crear_consumo`)
*   **`factor_estacional`** (Categórica): Estación del año en la que se registra la medición ('Verano', 'Invierno', etc.).
*   **`temperatura_media`** (Flotante): Temperatura climática asignada según la estación (con ruido estocástico).
*   **`consumo_kwh`** (Flotante): **Variable principal de análisis:** Consumo total simulado en kilovatios-hora.
*   **`uso_horario_pico`** (Booleano): Indica si el mayor consumo se da en franjas de alta demanda.
*   **`horas_alto_consumo`** (Entero): Franja horaria de mayor demanda eléctrica de la propiedad.
*   **`tarifa_kwh`** (Flotante): Precio variable del kWh en USD.
*   **`costo_estimado`** (Flotante): Facturación estimada (Consumo multiplicado por Tarifa). *Nota: Altamente correlacionada con el consumo.*

###  Puntuación y Variable Objetivo (`crear_score_y_target`)
*   **`energy_efficiency_score`** (Flotante): Puntuación determinística (0-100) basada en la eficiencia de la infraestructura y el consumo per cápita.
*   **`categoria`** (Categórica): **Variable Target (Clasificación):** Etiqueta de eficiencia ('Efficient', 'Moderate', 'Inefficient'). Asignada por percentiles.

---

## 5. Diseño de Imperfecciones Inyectadas (Data Cleaning)
Para enriquecer el proceso de Ingeniería de Datos, el módulo `inyectar_imperfecciones` aplica ruido intencional al dataset base:
1.  **Valores Nulos (NaN):** Inyectados de forma aleatoria (MCAR - *Missing Completely At Random*) entre un 2% y 6% en columnas clave como `ingreso_mensual`, `antiguedad_vivienda`, `electrodomesticos_eficientes` y `horas_en_casa`.
2.  **Outliers Extremos (Mansiones):** Simulación de propiedades inusualmente grandes, modificando el 0.5% de los datos con superficies entre 500m² y 1200m².
3.  **Outliers Extremos (Anomalías Energéticas):** Simulación de consumos absurdos (ej. Minería Cripto clandestina), multiplicando aleatoriamente el consumo de algunas viviendas por factores de 4 a 8.
4.  **Inconsistencias Lógicas:** Creación de registros contradictorios, como propiedades con excelente aislamiento y paneles solares, pero con un consumo inexplicablemente altísimo, requiriendo investigación en el EDA.
5.  **Registros Duplicados:** 1% del dataset copiado de manera idéntica y con reasignación de ID para forzar una limpieza basada en el análisis de características estructurales repetidas.

In [ ]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURACIÓN GLOBAL
# ==========================================
SEED = 42
N_REGISTROS = 10000

np.random.seed(SEED)
random.seed(SEED)

# ==========================================
# MÓDULOS DE GENERACIÓN DE DATOS
# ==========================================

def crear_viviendas(n: int) -> pd.DataFrame:
    """
    Genera las características estructurales de las viviendas.
    Utiliza distribuciones log-normales para áreas (evitando colas largas irreales)
    y poisson para conteo de habitaciones.
    """
    df = pd.DataFrame({'id': range(1, n + 1)})

    # 1. Tipo de vivienda (Distribución categórica)
    df['tipo_vivienda'] = np.random.choice(
        ['Casa', 'Departamento', 'Pequeño Comercio'],
        size=n, p=[0.55, 0.40, 0.05]
    )

    # 2. Metros Cuadrados (Log-normal acotada por tipo)
    def generar_m2(tipo):
        if tipo == 'Casa':
            return np.random.lognormal(mean=np.log(120), sigma=0.4)
        elif tipo == 'Departamento':
            return np.random.lognormal(mean=np.log(65), sigma=0.3)
        else:
            return np.random.lognormal(mean=np.log(90), sigma=0.5)

    df['metros_cuadrados'] = df['tipo_vivienda'].apply(generar_m2)
    df['metros_cuadrados'] = np.clip(df['metros_cuadrados'], 30, 450).round(1)

    # 3. Habitaciones y Baños (Poisson correlacionado al tamaño)
    # 1 habitación por cada ~35m2 + base
    lambda_hab = df['metros_cuadrados'] / 35
    df['habitaciones'] = np.random.poisson(lam=lambda_hab)
    df['habitaciones'] = np.clip(df['habitaciones'], 1, 8).astype(int)

    # 1 baño por cada 2-3 habitaciones
    df['baños'] = np.random.poisson(lam=(df['habitaciones'] / 2 + 0.5))
    df['baños'] = np.clip(df['baños'], 1, 5).astype(int)

    # 4. Antigüedad (Distribución Triangular, la mayoría tiene ~15 años)
    df['antiguedad_vivienda'] = np.random.triangular(left=0, mode=15, right=80, size=n).astype(int)

    # 5. Aislamiento y Eficiencia (Correlacionado inversamente con antigüedad)
    # Viviendas más nuevas tienden a tener mejor aislamiento
    prob_base = np.clip(1 - (df['antiguedad_vivienda'] / 80), 0.1, 0.9)

    aislamiento_choices = ['Poor', 'Average', 'Good', 'Excellent']
    df['aislamiento'] = [
        np.random.choice(aislamiento_choices, p=[1-p, (1-p)*0.5, p*0.6, p*0.4] / np.sum([1-p, (1-p)*0.5, p*0.6, p*0.4]))
        for p in prob_base
    ]

    eficiencia_choices = ['E', 'D', 'C', 'B', 'A']
    df['eficiencia_construccion'] = [
        np.random.choice(eficiencia_choices, p=[1-p, (1-p)*0.5, 0.2, p*0.5, p*0.5] / np.sum([1-p, (1-p)*0.5, 0.2, p*0.5, p*0.5]))
        for p in prob_base
    ]

    # 6. Paneles Solares (Binomial: Mayor probabilidad en casas nuevas y grandes)
    prob_solar = np.where(df['tipo_vivienda'] == 'Casa', 0.08, 0.01)
    prob_solar += np.where(df['antiguedad_vivienda'] < 10, 0.05, 0.0)
    df['paneles_solares'] = np.random.binomial(1, np.clip(prob_solar, 0.0, 0.2)).astype(bool)

    return df

def crear_habitantes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera información demográfica y ocupacional fuertemente atada
    a las características del inmueble.
    """
    n = len(df)

    # 1. Cantidad de personas (Poisson condicionado por habitaciones)
    lambda_personas = np.where(df['tipo_vivienda'] == 'Pequeño Comercio',
                               df['habitaciones'] * 1.5,
                               df['habitaciones'] * 0.8 + 0.5)
    df['cantidad_personas'] = np.random.poisson(lam=lambda_personas)
    df['cantidad_personas'] = np.clip(df['cantidad_personas'], 1, 8).astype(int)

    # 2. Trabajo Remoto (Binomial)
    # Comercitos no tienen "trabajo remoto" per se en este contexto
    prob_remoto = np.where(df['tipo_vivienda'] == 'Pequeño Comercio', 0.0, 0.35)
    df['trabajo_remoto'] = np.random.binomial(1, prob_remoto).astype(bool)

    # 3. Horas en casa (Distribución normal ajustada por remoto/tipo)
    base_horas = np.random.normal(loc=12, scale=3, size=n)
    horas = np.where(df['trabajo_remoto'], base_horas + 8, base_horas)
    df['horas_en_casa'] = np.clip(horas, 8, 24).round(1)

    # 4. Ingreso Mensual (Distribución Log-normal, correlacionado con m2 y personas)
    # Simulamos valores en dólares (USD) para estandarización.
    base_ingreso = np.random.lognormal(mean=np.log(1500), sigma=0.6, size=n)
    multiplicador_m2 = df['metros_cuadrados'] / df['metros_cuadrados'].mean()
    df['ingreso_mensual'] = (base_ingreso * multiplicador_m2).round(2)

    return df

def crear_equipamiento(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera el parque de electrodomésticos basándose en poder adquisitivo,
    tamaño del inmueble y cantidad de habitantes.
    """
    n = len(df)

    # Aires acondicionados (Correlación con m2 e ingresos)
    prob_ac = (df['metros_cuadrados'] / 100) * (df['ingreso_mensual'] / 2000)
    df['aires_acondicionados'] = np.random.poisson(lam=np.clip(prob_ac, 0.5, 3))
    df['aires_acondicionados'] = np.clip(df['aires_acondicionados'], 0, 5).astype(int)

    # Heladeras (Al menos 1, más si hay mucha gente/comercios)
    df['heladeras'] = 1 + np.random.binomial(1, p=np.clip(df['cantidad_personas']/10, 0, 0.5))
    df['heladeras'] = np.where(df['tipo_vivienda'] == 'Pequeño Comercio', df['heladeras'] + 1, df['heladeras'])

    # Computadoras y TVs
    df['televisores'] = np.random.poisson(lam=df['cantidad_personas'] * 0.7)
    df['computadoras'] = np.random.poisson(lam=df['cantidad_personas'] * 0.8)
    df['computadoras'] = np.where(df['trabajo_remoto'], df['computadoras'] + 1, df['computadoras'])

    # Lavado
    df['lavadoras'] = np.where(df['tipo_vivienda'] != 'Pequeño Comercio', 1, 0)
    df['secadoras'] = np.random.binomial(1, p=np.clip(df['ingreso_mensual']/10000, 0.05, 0.4))

    df['cantidad_equipos'] = (df['aires_acondicionados'] + df['heladeras'] +
                              df['televisores'] + df['computadoras'] +
                              df['lavadoras'] + df['secadoras'] +
                              np.random.poisson(lam=5, size=n)) # Pequeños electrodomésticos

    # Calefacción y climatización
    df['calefaccion'] = np.random.choice([True, False], size=n, p=[0.8, 0.2])

    def asignar_tipo_calefaccion(row):
        if not row['calefaccion']: return 'Ninguna'
        if row['antiguedad_vivienda'] > 30: return 'Gas'
        if row['ingreso_mensual'] > 4000: return 'Losa Radiante'
        return 'Eléctrica'

    df['tipo_calefaccion'] = df.apply(asignar_tipo_calefaccion, axis=1)

    # Iluminación
    df['tipo_iluminacion'] = np.where(
        df['antiguedad_vivienda'] < 10,
        np.random.choice(['LED', 'Mixta'], size=n, p=[0.8, 0.2]),
        np.random.choice(['LED', 'Mixta', 'Incandescente'], size=n, p=[0.3, 0.5, 0.2])
    )

    # Eficiencia de electrodomésticos (%)
    # Normal distribution, negatively correlated with home age.
    base_eff = 80 - (df['antiguedad_vivienda'] * 0.5) + (df['ingreso_mensual'] / 1000)
    df['electrodomesticos_eficientes'] = np.random.normal(loc=base_eff, scale=10)
    df['electrodomesticos_eficientes'] = np.clip(df['electrodomesticos_eficientes'], 10, 100).round(1)

    return df

def crear_consumo(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula el consumo eléctrico basándose en las variables anteriores,
    aplicando ruido estocástico (Gaussian noise) para realismo.
    """
    n = len(df)

    # Estacionalidad
    estaciones = ['Verano', 'Invierno', 'Primavera', 'Otoño']
    df['factor_estacional'] = np.random.choice(estaciones, size=n)

    # Temperatura Media (Dependiente de la estación + ruido)
    temp_dict = {'Verano': 28, 'Invierno': 8, 'Primavera': 20, 'Otoño': 15}
    df['temperatura_media'] = df['factor_estacional'].map(temp_dict) + np.random.normal(0, 3, size=n)
    df['temperatura_media'] = df['temperatura_media'].round(1)

    # Consumo Base Físico
    consumo_base = (df['metros_cuadrados'] * 0.8) + (df['cantidad_personas'] * 45)

    # Consumo de Climatización (Penalizado por mal aislamiento)
    factor_aislamiento = df['aislamiento'].map({'Poor': 1.4, 'Average': 1.1, 'Good': 0.8, 'Excellent': 0.6})

    consumo_frio = np.where((df['factor_estacional'] == 'Verano') & (df['aires_acondicionados'] > 0),
                            df['aires_acondicionados'] * 120 * factor_aislamiento, 0)

    consumo_calor = np.where((df['factor_estacional'] == 'Invierno') & (df['tipo_calefaccion'] == 'Eléctrica'),
                             180 * factor_aislamiento, 0)

    # Consumo Equipamiento
    factor_iluminacion = df['tipo_iluminacion'].map({'LED': 0.6, 'Mixta': 1.0, 'Incandescente': 1.8})
    consumo_equipos = (df['cantidad_equipos'] * 25) * factor_iluminacion * (1 - (df['electrodomesticos_eficientes'] * 0.003))

    # Suma de consumos + Impacto de horas en casa
    consumo_total = (consumo_base + consumo_frio + consumo_calor + consumo_equipos) * (df['horas_en_casa'] / 16)

    # Descuento por paneles solares (Reduce entre 20% y 60% del consumo de red)
    descuento_solar = np.where(df['paneles_solares'], np.random.uniform(0.2, 0.6, size=n), 0)
    consumo_total = consumo_total * (1 - descuento_solar)

    # Ruido Aleatorio (Añade imperfección humana)
    ruido = np.random.normal(1.0, 0.15, size=n)
    df['consumo_kwh'] = np.clip(consumo_total * ruido, 50, None).round(2)

    # Variables de Uso
    df['uso_horario_pico'] = np.where(df['trabajo_remoto'],
                                      np.random.binomial(1, 0.7),
                                      np.random.binomial(1, 0.4)).astype(bool)

    df['horas_alto_consumo'] = np.where(df['uso_horario_pico'],
                                        np.random.normal(19, 2, size=n),
                                        np.random.normal(14, 4, size=n))
    df['horas_alto_consumo'] = np.clip(df['horas_alto_consumo'], 0, 23).astype(int)

    # Variables Económicas
    df['tarifa_kwh'] = np.random.normal(0.18, 0.02, size=n).round(3) # Precio en USD/kWh con variaciones
    df['costo_estimado'] = (df['consumo_kwh'] * df['tarifa_kwh']).round(2)

    return df

def crear_score_y_target(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula un "Energy Efficiency Score" transparente y determinístico,
    y a partir de ahí asigna la variable objetivo desbalanceada.
    """
    # Índices parciales (Max ~ 100)
    idx_aislamiento = df['aislamiento'].map({'Poor': 0, 'Average': 10, 'Good': 20, 'Excellent': 30})
    idx_const = df['eficiencia_construccion'].map({'E': 0, 'D': 5, 'C': 10, 'B': 15, 'A': 20})
    idx_paneles = np.where(df['paneles_solares'], 15, 0)
    idx_electro = (df['electrodomesticos_eficientes'] / 100) * 20
    idx_ilum = df['tipo_iluminacion'].map({'Incandescente': 0, 'Mixta': 7, 'LED': 15})

    # Penalización por consumo excesivo per cápita
    consumo_per_capita = df['consumo_kwh'] / df['cantidad_personas']
    penalizacion = np.clip((consumo_per_capita - 150) / 10, 0, 30)

    score_crudo = idx_aislamiento + idx_const + idx_paneles + idx_electro + idx_ilum - penalizacion

    # Añadir un leve ruido al score (para que el modelo ML tenga que esforzarse)
    score_final = score_crudo + np.random.normal(0, 5, size=len(df))
    df['energy_efficiency_score'] = np.clip(score_final, 0, 100).round(1)

    # Target: Asignación por percentiles para forzar el desbalance deseado (~30% / 45% / 25%)
    p70 = np.percentile(df['energy_efficiency_score'], 70) # Top 30%
    p25 = np.percentile(df['energy_efficiency_score'], 25) # Bottom 25%

    def clasificar(score):
        if score >= p70: return 'Efficient'
        elif score <= p25: return 'Inefficient'
        else: return 'Moderate'

    df['categoria'] = df['energy_efficiency_score'].apply(clasificar)

    return df

def inyectar_imperfecciones(df: pd.DataFrame) -> pd.DataFrame:
    """
    Inyecta ruido sucio, missing values (NaN), outliers e inconsistencias
    necesarias para la fase de Data Cleaning y EDA.
    """
    df_dirty = df.copy()
    n = len(df_dirty)

    # 1. Valores Faltantes (MCAR - Missing Completely At Random)
    cols_na = ['ingreso_mensual', 'antiguedad_vivienda', 'electrodomesticos_eficientes', 'horas_en_casa']
    for col in cols_na:
        prop_na = np.random.uniform(0.02, 0.06)
        mask = np.random.choice([True, False], size=n, p=[prop_na, 1 - prop_na])
        df_dirty.loc[mask, col] = np.nan

    # 2. Outliers (Casos extremos pero plausibles)
    # Mansiones enormes
    idx_out_m2 = np.random.choice(df_dirty.index, size=int(n * 0.005), replace=False)
    df_dirty.loc[idx_out_m2, 'metros_cuadrados'] = np.random.uniform(500, 1200, size=len(idx_out_m2))

    # Granjas de servidores / Minería cripto escondida (Consumo absurdo)
    idx_out_cons = np.random.choice(df_dirty.index, size=int(n * 0.005), replace=False)
    df_dirty.loc[idx_out_cons, 'consumo_kwh'] = df_dirty.loc[idx_out_cons, 'consumo_kwh'] * np.random.uniform(4, 8)

    # 3. Inconsistencias Lógicas Controladas
    # Casas 'Excellent' pero que consumen excesivamente
    idx_inc = np.random.choice(df_dirty[(df_dirty['aislamiento'] == 'Excellent') &
                                        (df_dirty['paneles_solares'] == True)].index,
                               size=int(n * 0.002), replace=False)
    df_dirty.loc[idx_inc, 'consumo_kwh'] = np.random.uniform(2000, 3500, size=len(idx_inc))

    # 4. Duplicados
    num_duplicados = int(n * 0.01)
    df_duplicados = df_dirty.sample(n=num_duplicados, random_state=SEED)
    df_dirty = pd.concat([df_dirty, df_duplicados]).reset_index(drop=True)

    # Mezclamos el dataset (Shuffle)
    df_dirty = df_dirty.sample(frac=1, random_state=SEED).reset_index(drop=True)
    df_dirty['id'] = range(1, len(df_dirty) + 1) # Rehacer IDs para no delatar los duplicados fácilmente

    return df_dirty

def main():
    print(f"🚀 Iniciando generación de dataset de Consumo Energético ({N_REGISTROS} registros)...")

    # Ejecutar pipeline
    df = crear_viviendas(N_REGISTROS)
    df = crear_habitantes(df)
    df = crear_equipamiento(df)
    df = crear_consumo(df)
    df = crear_score_y_target(df)
    df_final = inyectar_imperfecciones(df)

    # Exportar a CSV
    filename = 'dataset_consumo_energetico.csv'
    df_final.to_csv(filename, index=False, encoding='utf-8')

    # ==========================================
    # REPORTING Y ESTADÍSTICAS
    # ==========================================
    print(f"\n✅ Dataset generado y guardado como: '{filename}'")
    print("-" * 50)
    print("📊 RESUMEN DE LOS DATOS GENERADOS")
    print("-" * 50)
    print(f"🔹 Número total de registros (con duplicados): {len(df_final)}")
    print(f"🔹 Número de columnas: {df_final.shape[1]}")
    print(f"🔹 Duplicados intencionales agregados: {len(df_final) - N_REGISTROS}")
    print("\n⚠️ Valores faltantes inyectados por columna:")
    missing = df_final.isnull().sum()
    print(missing[missing > 0].to_string())

    print("\n🎯 Distribución de la Variable Objetivo (Categoría):")
    print(df_final['categoria'].value_counts(normalize=True).mul(100).round(2).astype(str) + ' %')

    print("\n💰 Estadísticas Descriptivas (Muestra reducida):")
    cols_desc = ['metros_cuadrados', 'cantidad_personas', 'consumo_kwh', 'costo_estimado']
    print(df_final[cols_desc].describe().round(2))

    print("\n🔗 Matriz de Correlación (Top variables numéricas):")
    cols_corr = ['metros_cuadrados', 'habitaciones', 'aires_acondicionados', 'consumo_kwh', 'energy_efficiency_score']
    print(df_final[cols_corr].corr().round(3))

    print("-" * 50)
    print("¡Listo! Tienes un dataset con distribuciones reales, correlaciones latentes, ruido estadístico y data sucia lista para EDA y Machine Learning.")

if __name__ == '__main__':
    main()

🚀 Iniciando generación de dataset de Consumo Energético (10000 registros)...

✅ Dataset generado y guardado como: 'dataset_consumo_energetico.csv'
--------------------------------------------------
📊 RESUMEN DE LOS DATOS GENERADOS
--------------------------------------------------
🔹 Número total de registros (con duplicados): 10100
🔹 Número de columnas: 33
🔹 Duplicados intencionales agregados: 100

⚠️ Valores faltantes inyectados por columna:
antiguedad_vivienda             461
horas_en_casa                   289
ingreso_mensual                 438
electrodomesticos_eficientes    278

🎯 Distribución de la Variable Objetivo (Categoría):
categoria
Moderate       44.92 %
Efficient      30.01 %
Inefficient    25.07 %
Name: proportion, dtype: object

💰 Estadísticas Descriptivas (Muestra reducida):
       metros_cuadrados  cantidad_personas  consumo_kwh  costo_estimado
count          10100.00           10100.00     10100.00        10100.00
mean             108.56               3.06       498